In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
data = pd.read_csv("data.csv")
data = data[['R','G', 'B', 'L_cal', 'a_cal', 'b_cal', 'L*', 'a*', 'b*', 'VDO', 'File_Name', 'Crop_Index']]

# 1. normalize 'R', 'G', 'B' (min = 0, max = 255) to [0,1]
data[['R', 'G', 'B']] = data[['R', 'G', 'B']] / 255.0

# 2. normalize 'L_cal', 'L*' (min = 0, max = 100) to [0,1]
data[['L_cal', 'L*']] = data[['L_cal', 'L*']] / 100.0

# 3. normalize 'a_cal', 'b_cal', 'a*', 'b*' (min = -120, max = 120) to [0,1]
# สูตร Min-Max Scaling: (X - Min) / (Max - Min)
# ดังนั้น: (X - (-120)) / (120 - (-120)) = (X + 120) / 240
ab_cols = ['a_cal', 'b_cal', 'a*', 'b*']
data[ab_cols] = (data[ab_cols] + 120.0) / 240.0

# ตรวจสอบผลลัพธ์
print(data.describe()) # ดูค่า min, max ของแต่ละคอลัมน์เพื่อความชัวร์

In [ ]:
print(data)

In [ ]:
# สร้างคอลัมน์ 'Base_Color' โดยตัดตัวเลข _1, _2, _3 ด้านหลังออก
# เช่น 'M_136_1' จะกลายเป็น 'M_136'
data['Base_Color'] = data['VDO'].str.rsplit('_', n=1).str[0]


# แบ่ง train test โดย data ที่ column Base_Color มีค่าเดียวกันอยู่ กลุ่มเดียวกัน
# 1. ดึงรายชื่อ Base_Color ทั้งหมดที่ไม่ซ้ำกัน
unique_vdos = data['Base_Color'].unique()

# 2. สับเปลี่ยนลำดับ (Shuffle) รายชื่อ Base_Color เพื่อความสุ่ม
np.random.seed(42)
np.random.shuffle(unique_vdos)

# 3. กำหนดจุดตัดแบ่งข้อมูล (เช่น Train 80%, Test 20%)
split_index = int(len(unique_vdos) * 0.8)

# 4. แบ่งรายชื่อ VDO ออกเป็น 2 กลุ่ม
train_vdo_names = unique_vdos[:split_index]
test_vdo_names = unique_vdos[split_index:]

# 5. กรองข้อมูลจาก DataFrame เดิม
first_data = data[data['Base_Color'].isin(train_vdo_names)].copy()
sec_data = data[data['Base_Color'].isin(test_vdo_names)].copy()

print(f"first_data set: มี {len(first_data)} rows จาก {len(train_vdo_names)} Base_Color")
print(first_data['VDO'].unique())

print(f"sec_data set: มี {len(sec_data)} rows จาก {len(test_vdo_names)} Base_Color")
print(sec_data['VDO'].unique())

In [ ]:
train_labels = first_data[['L*','a*','b*']]
train_data = first_data.drop(columns=['L*','a*','b*','VDO','File_Name','Crop_Index','Base_Color'])
# สร้าง columns R*G, R*B, G*B, R**2, G**2, B**2
train_data['R*G'] = train_data['R'] * train_data['G']
train_data['R*B'] = train_data['R'] * train_data['B']
train_data['G*B'] = train_data['G'] * train_data['B']

train_data['R**2'] = train_data['R'] ** 2
train_data['G**2'] = train_data['G'] ** 2
train_data['B**2'] = train_data['B'] ** 2

# ตรวจสอบผลลัพธ์
print(train_data.head())

In [ ]:
test_labels = sec_data[['L*','a*','b*']]
test_data = sec_data.drop(columns=['L*','a*','b*','VDO','File_Name','Crop_Index','Base_Color'])
# สร้าง columns R*G, R*B, G*B, R**2, G**2, B**2
test_data['R*G'] = test_data['R'] * test_data['G']
test_data['R*B'] = test_data['R'] * test_data['B']
test_data['G*B'] = test_data['G'] * test_data['B']

test_data['R**2'] = test_data['R'] ** 2
test_data['G**2'] = test_data['G'] ** 2
test_data['B**2'] = test_data['B'] ** 2

# ตรวจสอบผลลัพธ์
print(test_data.head())

In [ ]:
train_data.shape[1]

In [ ]:
print(len(train_data))
print(len(test_data))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# เลือกเฉพาะคอลัมน์ที่เป็น Feature ของ Quadratic Model (รวม 12 ตัว) ['R', 'G', 'B', 'L_cal', 'a_cal', 'b_cal', 'R*G', 'R*B', 'G*B', 'R**2', 'G**2', 'B**2']

X_train = train_data
y_train = train_labels

X_test = test_data
y_test = test_labels

# สร้างโครงสร้าง Neural Network ตาม Paper
model = Sequential([
    # Input layer รับค่าจาก 12 Quadratic Features
    Input(shape=(X_train.shape[1],)),
    # Hidden layer: ใช้ 2 ชั้นและ 32,16
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    # Output layer: 3 นิวรอน สำหรับค่า L*, a*, b* (ใช้ linear activation เพราะเป็น Regression)
    Dense(3, activation='linear')
])

# คอมไพล์โมเดล โดยใช้ loss เป็น Mean Absolute Error (MAE) ตามสมการในเปเปอร์
model.compile(optimizer=Adam(learning_rate=0.001),loss='mae',metrics=['mae', 'mse'])

# สรุปโครงสร้างโมเดล
model.summary()

# ตั้งค่า Early Stopping เพื่อหยุดเทรนเมื่อ Validation Loss ไม่ลดลง (ตามเปเปอร์)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=40, # หาก error บน validation set ไม่ลดลงติดต่อกัน 40 epochs ให้หยุด
    restore_best_weights=True
)

# เริ่มการเทรนโมเดล
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=500, # ตั้งเผื่อไว้ Early Stopping จะหยุดให้เอง
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# ประเมินผลลัพธ์กับ Test Set
test_loss, test_mae, test_mse = model.evaluate(X_test, y_test)
print(f"Test MAE: {test_mae:.4f}")

In [ ]:
import numpy as np
model = tf.keras.models.load_model('my_model.keras')
# 1. ทำนายผลลัพธ์จาก Test set ทั้งหมดในรวดเดียว (ไม่ต้องใช้ loop)
preds_norm = model.predict(X_test)

# 2. Denormalize ผลทำนายและผลจริง กลับเป็นสเกลปกติ
# L* สเกลเดิมคือ 0 ถึง 100 (ตอนแปลงเราหาร 100)
preds_L = preds_norm[:, 0] * 100.0
trues_L = y_test['L*'].values * 100.0

# a* และ b* สเกลเดิมคือ -120 ถึง 120 (ตอนแปลงเราทำ (x+120)/240)
preds_a = (preds_norm[:, 1] * 240.0) - 120.0
trues_a = (y_test['a*'].values * 240.0) - 120.0

preds_b = (preds_norm[:, 2] * 240.0) - 120.0
trues_b = (y_test['b*'].values * 240.0) - 120.0

# นำกลับมารวมเป็น Array เดียวกัน
preds_all = np.column_stack((preds_L, preds_a, preds_b))
trues_all = np.column_stack((trues_L, trues_a, trues_b))

# 3. คำนวณค่า RMSE
rmse_L = np.sqrt(((preds_all[:,0] - trues_all[:,0])**2).mean())
rmse_a = np.sqrt(((preds_all[:,1] - trues_all[:,1])**2).mean())
rmse_b = np.sqrt(((preds_all[:,2] - trues_all[:,2])**2).mean())
rmse_total = np.sqrt(((preds_all - trues_all)**2).mean())

# 4. คำนวณ Error (Mean Normalized Error) ตามสมการใน Paper
e_L = np.abs(preds_all[:,0] - trues_all[:,0]).mean() / 100.0
e_a = np.abs(preds_all[:,1] - trues_all[:,1]).mean() / 240.0
e_b = np.abs(preds_all[:,2] - trues_all[:,2]).mean() / 240.0

# คำนวณ Total Error เป็นเปอร์เซ็นต์
e_total = ((e_L + e_a + e_b) / 3.0) * 100.0

# 5. แสดงผลลัพธ์เปรียบเทียบ
print(f"\nNN(Quad+Lab) RMSE  L:{rmse_L:.2f}  a:{rmse_a:.2f}  b:{rmse_b:.2f}  total:{rmse_total:.2f}")
print(f"NN(Quad+Lab) Error (paper style): {e_total:.2f}%\n")

print("--- เทียบกับผลลัพธ์อ้างอิง ---")
print(f"Paper Quadratic (RGB only):         1.23%") # อ้างอิงจาก Table 3 ใน Paper
print(f"Paper NN (RGB only):                0.93%") # อ้างอิงจาก Table 3 ใน Paper

In [ ]:
# Save Model Tensor
# Save the entire model 
model.save('my_model.keras')
# Load the model back later
loaded_model = tf.keras.models.load_model('my_model.keras')


32-16 : 1.09%